# ⚡ New Energy Agent · Transformers (stable ~25 tok/s)
**Colab Free T4 GPU · Runtime → Run all**

| Component | Version | Notes |
|-----------|---------|-------|
| GPU | T4 15GB | Turing SM75 |
| Inference | **transformers + autoawq** | Direct, no server |
| Model | Qwen2.5-3B-AWQ | ~3.5 GB VRAM |

### Data Persistence
| Data | Drive | Survives |
|------|-------|----------|
| Model (~2.5GB) | `/hf_cache` | ✅ |
| Electricity DB | `/new-energy-data` | ✅ |
| pip packages | Local SSD | ❌ (fast) |

### Usage
**Runtime → Run all** → Allow Drive → Get URL

⚡ For faster speed: open `colab_notebook.ipynb` (vLLM)


In [ ]:
# Cell 1: Read Secrets
import os
try:
    from google.colab import userdata
    for name in ['HF_TOKEN', 'NGROK_TOKEN']:
        try:
            val = userdata.get(name)
            if val:
                os.environ[name] = val
                print(f'OK {name}')
            else:
                print(f'Skip {name}')
        except: print(f'Skip {name}')
except ImportError:
    print('Not Colab')
print('Done')


In [ ]:
# Cell 2: Mount Google Drive
import os, shutil
mp = '/content/drive'
if os.path.isdir(mp) and os.listdir(mp):
    if os.path.isdir(os.path.join(mp, 'MyDrive')):
        print('Already mounted')
    else:
        for item in os.listdir(mp):
            p = os.path.join(mp, item)
            try: (shutil.rmtree if os.path.isdir(p) else os.remove)(p)
            except: pass
        from google.colab import drive; drive.mount(mp)
else:
    from google.colab import drive; drive.mount(mp)

DIRS = {
    'hf': '/content/drive/MyDrive/hf_cache',
    'data': '/content/drive/MyDrive/new-energy-data',
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = DIRS['hf']
os.environ['HF_HUB_CACHE'] = DIRS['hf']
os.environ['NEW_ENERGY_DATA_DIR'] = DIRS['data']

import sqlite3
db = os.path.join(DIRS['data'], 'electricity_cache.db')
if os.path.exists(db):
    n = sqlite3.connect(db).execute('SELECT COUNT(*) FROM electricity_prices').fetchone()[0]
    print(f'Electricity cache: {n} records')
print('Drive ready')


In [ ]:
# Cell 3: Install autoawq + transformers + gradio (pip to local SSD)
import subprocess, sys
print('Installing packages (~2 GB)...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'autoawq', 'accelerate', 'transformers',
    'gradio', 'plotly', 'pandas', 'duckduckgo_search',
    'pyngrok', 'huggingface_hub', 'httpx', 'requests'
], check=False)
print('Done')


In [ ]:
# Cell 4: Clone repo
import os
rd = '/content/new-energy-agent'
if os.path.isdir(rd):
    %cd {rd}
    !git pull -q
else:
    !git clone -q https://github.com/pai-pixel/new-energy-agent.git {rd}
    %cd {rd}
print(f'Repo: {os.getcwd()}')


In [ ]:
# Cell 5: Download model to Drive (~2.5 GB, first time only)
import os, glob, time
from huggingface_hub import snapshot_download

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct-AWQ'
CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(CACHE, exist_ok=True)

hub = os.path.join(CACHE, 'hub')
found = None
if os.path.isdir(hub):
    dirs = glob.glob(os.path.join(hub, 'models--Qwen*', 'snapshots', '*'))
    for d in dirs:
        if os.path.isdir(d) and os.listdir(d):
            found = d; break

if found:
    gb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(found) for f in fs) / 1e9
    print(f'Model cached: {found} ({gb:.1f} GB)')
else:
    print(f'Downloading {MODEL_ID}...')
    t0 = time.time()
    found = snapshot_download(MODEL_ID, cache_dir=CACHE, resume_download=True, max_workers=4)
    print(f'Done ({time.time()-t0:.0f}s)')

os.environ['MODEL_PATH'] = found
print(f'Model: {found}')


In [ ]:
# Cell 6: Check GPU + Load model
import sys, os
sys.path.insert(0, '/content/new-energy-agent')

gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
if not gpu: print('No GPU! Runtime -> T4'); raise SystemExit(1)
print(f'GPU: {gpu[0]}')

from src.model_engine import load_model
model_path = os.environ.get('MODEL_PATH')
print(f'Loading model (first time compiles AWQ kernels ~30 sec)...')
load_model(model_path)
print('Model loaded!')


In [ ]:
# Cell 7: Start Agent (transformers fallback backend)
import os, sys
%cd /content/new-energy-agent
sys.path.insert(0, '/content/new-energy-agent')

# Override default: use transformers instead of vLLM
os.environ['INFERENCE_BACKEND'] = 'transformers'

import gradio as gr
from src.agent import NewEnergyAgent, create_ui, start_ngrok
from IPython.display import display, Javascript
import logging
logging.basicConfig(level=logging.WARNING)

ngrok_url = start_ngrok(7860)
agent = NewEnergyAgent()
demo = create_ui(agent)

print('=' * 50)
print('  New Energy Agent Ready! (transformers ~25 tok/s)')
print('=' * 50)
if ngrok_url:
    print(f'  ngrok: {ngrok_url}')
    display(Javascript(f'window.open("{ngrok_url}", "_blank");'))
print('  Gradio URL: see cell output for .gradio.live')
print('=' * 50)

demo.queue(max_size=32).launch(
    server_name='0.0.0.0', server_port=7860,
    share=True, show_error=True,
    css='.gradio-container{max-width:900px!important}',
    theme=gr.themes.Soft(primary_hue='green'),
)


In [ ]:
# Cell 8 (optional): Keep-alive
from IPython.display import display, Javascript
display(Javascript('setInterval(function(){document.querySelector("colab-connect-button").click()},60000)'))
import os, sqlite3
db = os.path.join(os.environ.get('NEW_ENERGY_DATA_DIR','/content/drive/MyDrive/new-energy-data'), 'electricity_cache.db')
if os.path.exists(db):
    c = sqlite3.connect(db)
    t = c.execute('SELECT COUNT(*) FROM electricity_prices').fetchone()[0]
    ps = c.execute('SELECT DISTINCT province FROM electricity_prices').fetchall()
    c.close()
    print(f'Cache: {t} records | provinces: {", ".join(p[0] for p in ps)}')
print('Keep-alive active.')


---
### Usage
- `Shanghai feed-in tariff` / `Jiangsu desulfurized coal price`
- `Beijing weather` / `Solar subsidy policy`
- Multi-turn: `Shanghai feed-in` -> `Jiangsu too` -> `commercial instead`

### For faster speed
Switch to vLLM backend: open `colab_notebook.ipynb`

[GitHub](https://github.com/pai-pixel/new-energy-agent)